# Метамодель с функцией потерь соревнования

Отдельный эксперимент на готовых прогнозах: контроль CatBoost depth 4 + Ridge против линейной модели и MLP с 16 скрытыми нейронами. Для каждой новой архитектуры сравниваются MAE и WAPE + |Relative Bias|. Функция потерь считается по всему обучающему периоду, со всеми маршрутами и горизонтами; это не средняя дневная метрика. Поправка к взвешенной смеси ограничена ±5%, штраф стягивает её к нулю. Число эпох и настройки фиксированы заранее.

Подключите только **полный распакованный output blend-best-2** через Add Input. Базовые модели повторно не обучаются. Запуск на CPU, GPU не требуется. Обучение запускает пользователь в Kaggle.

Выбор по среднему score на неделях 4–10 и 11–17 октября. Каждому окну предшествует своё обучение. После фиксации выбора на test 18–31 октября оцениваются только выбранный вариант и контроль. Метамодели обучаются на OOF до test. Этот test уже изучался: результат является проверкой гипотезы на прежнем периоде, а не новым holdout.

В таблицах train_oof — метамодель на собственном обучении, val/test — следующий временной период. Сохраняются WAPE, Relative Bias, итоговый score и дневные срезы. Результаты в `/kaggle/working/blend_metric_*`. Сабмит не собирается: сначала проверяем гипотезу. Независимая от полного переобучения оценка test получается до переобучения.


In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display


HORIZONS = tuple(range(1, 9))
STEP = pd.Timedelta(minutes=30)
OOF_START = pd.Timestamp('2025-09-20')
LOCAL_TEST_START = pd.Timestamp('2025-10-18')
LOCAL_TEST_END = pd.Timestamp('2025-11-01')
OOF_WINDOWS = list(zip(pd.date_range(OOF_START, periods=4, freq='7D'),
                       pd.date_range(OOF_START + pd.Timedelta(days=7), periods=4, freq='7D')))
KEYS = ['cutoff', 'horizon', 'route_id']
GROUP_KEYS = KEYS[:2]
BASES = ['session6', 'rf', 'cat_rich', 'xgb_rich', 'xgb_clean', 'profile28']
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
SEED = 42
THREADS = 4


SESSION_FILES = ('comparison_meta.json', 'train_solo_track.parquet', 'test_solo_track.parquet',
                 'validation_predictions.parquet')
XGB_STAGES = ('oof_week_0', 'oof_week_1', 'oof_week_2', 'calibration_select', 'train_val')
XGB_FILES = ('manifest.json', 'protocol.json', 'evaluation_complete.json',
             *(f'{stage}_h{h}_xgb_k8.parquet' for stage in XGB_STAGES for h in HORIZONS))


def find_dataset(label, path, marker, required):
    if path is None:
        root = Path('/kaggle/input')
        candidates = sorted({file.parent for file in root.rglob(marker)})
        complete = [folder for folder in candidates if all((folder / name).is_file() for name in required)]
        if len(complete) == 1:
            return complete[0]
        if not complete and len(candidates) == 1:
            path = candidates[0]
        else:
            raise FileNotFoundError(f'{label}: нужен один полный набор. Укажите путь к папке в ячейке настроек. '
                                    f'Найденные полные наборы: {complete}; папки с {marker}: {candidates}')
    path = Path(path)
    missing = [name for name in required if not (path / name).is_file()]
    if missing:
        raise FileNotFoundError(f'{label}: в {path} отсутствуют файлы {missing}')
    return path


def digest_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path, data):
    temp = path.with_suffix('.partial')
    temp.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)


def save_parquet(path, data):
    temp = path.with_suffix('.partial')
    data.to_parquet(temp, index=False)
    temp.replace(path)


def initialize(session_path, xgb_path):
    global ARTIFACTS, TIMES, ROUTES, VALUES, SESSION_META, SESSION_DATA, XGB_DATA
    SESSION_DATA = find_dataset('Session6', session_path, 'comparison_meta.json', SESSION_FILES)
    XGB_DATA = find_dataset('XGBoost rich k8', xgb_path, 'oof_week_0_h1_xgb_k8.parquet', XGB_FILES)
    train_path = SESSION_DATA / 'train_solo_track.parquet'
    SESSION_META = json.loads((SESSION_DATA / 'comparison_meta.json').read_text())
    xgb_manifest = json.loads((XGB_DATA / 'manifest.json').read_text())
    XGB_PROTOCOL = json.loads((XGB_DATA / 'protocol.json').read_text())
    xgb_evaluation = json.loads((XGB_DATA / 'evaluation_complete.json').read_text())
    contract = dict(code=NOTEBOOK_HASH, seed=SEED,
                    session_sha256={name: digest_file(SESSION_DATA / name) for name in SESSION_FILES},
                    xgb_sha256={name: digest_file(XGB_DATA / name) for name in XGB_FILES})
    if (xgb_manifest['train_hash'] != contract['session_sha256']['train_solo_track.parquet'] or
            xgb_manifest['test_hash'] != contract['session_sha256']['test_solo_track.parquet']):
        raise ValueError('Датасеты XGBoost rich и Session6 используют разные входные файлы')
    if (pd.Timestamp(XGB_PROTOCOL['val_start']) != OOF_START or
            pd.Timestamp(XGB_PROTOCOL['calibration_split']) != OOF_WINDOWS[-1][0] or
            pd.Timestamp(XGB_PROTOCOL['test_start']) != LOCAL_TEST_START or
            pd.Timestamp(xgb_evaluation['train_val_model_target_end_exclusive']) != LOCAL_TEST_START or
            XGB_PROTOCOL['model_clusters'] != [8] or XGB_PROTOCOL['test_used_for_selection'] or
            not xgb_evaluation['test_evaluated_before_full_refit']):
        raise ValueError('Границы OOF или протокол архива XGB rich отличаются')
    run_id = hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()[:12]
    ARTIFACTS = Path('/kaggle/working') / f'best_blend_{run_id}'
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    write_json(ARTIFACTS / 'inputs.json', dict(contract, session_path=str(SESSION_DATA), xgb_path=str(XGB_DATA)))
    print('Session6:', SESSION_DATA)
    print('XGBoost rich k8:', XGB_DATA)
    history = pd.read_parquet(train_path, columns=['timestamp', 'route_id', TARGET, *STATUSES])
    history = history.sort_values(['timestamp', 'route_id'])
    TIMES = pd.DatetimeIndex(history.timestamp.unique())
    ROUTES = np.sort(history.route_id.unique())
    if (len(history) != len(TIMES) * len(ROUTES) or history.isna().any().any() or
            history.duplicated(['timestamp', 'route_id']).any() or
            not TIMES.equals(pd.date_range(TIMES[0], TIMES[-1], freq=STEP))):
        raise ValueError('Требуется полная получасовая панель')
    VALUES = {name: history[name].to_numpy(dtype=np.float32).reshape(len(TIMES), len(ROUTES))
              for name in (TARGET, *STATUSES)}
    if not all(np.isfinite(array).all() for array in VALUES.values()) or np.any(VALUES[TARGET] < 0):
        raise ValueError('Некорректные наблюдения исходной панели')
    display(pd.DataFrame([dict(split=f'oof_{i}', start=start, end_exclusive=stop)
                          for i, (start, stop) in enumerate(OOF_WINDOWS, 1)] +
                         [dict(split='test', start=LOCAL_TEST_START, end_exclusive=LOCAL_TEST_END)]))


def metric(y, prediction):
    actual = np.asarray(y, dtype=np.float64).ravel()
    predicted = np.asarray(prediction, dtype=np.float64).ravel()
    volume = actual.sum()
    if actual.shape != predicted.shape or not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError('Некорректный вход метрики')
    if volume == 0:
        return dict(rows=len(actual), WAPE=np.nan, relative_bias=np.nan, score=np.nan)
    error = predicted - actual
    wape = np.abs(error).sum() / volume
    bias = error.sum() / volume
    return dict(rows=len(actual), WAPE=wape, relative_bias=bias, score=wape + abs(bias))


def fit_report(model, boundary, horizon, group, split, train_y, train_pred, query_y, query_pred):
    report = pd.DataFrame([
        dict(model=model, fit_before=boundary, horizon=horizon, group=group, period=period, **metric(y, pred))
        for period, y, pred in [('train', train_y, train_pred), (split, query_y, query_pred)]
    ])
    display(report)
    path = ARTIFACTS / f'fits_{model}_{boundary:%Y%m%d}_h{horizon}_{group}.csv'
    report.to_csv(path, index=False)


def read_xgb(stage):
    if stage not in XGB_STAGES:
        raise ValueError(f'Неизвестный этап XGBoost rich: {stage}')
    frames = [pd.read_parquet(XGB_DATA / f'{stage}_h{h}_xgb_k8.parquet',
                              columns=['cutoff', 'timestamp', 'route_id', 'h', 'y', 'y_pred']) for h in HORIZONS]
    result = pd.concat(frames, ignore_index=True).rename(columns={'h': 'horizon', 'y_pred': 'xgb_rich'})
    if not result.timestamp.eq(result.cutoff + result.horizon * STEP).all():
        raise ValueError('Неверный timestamp в XGB rich')
    return result


def read_session6_test():
    frame = pd.read_parquet(SESSION_DATA / 'validation_predictions.parquet',
                            columns=KEYS + ['y', 'session_prediction'])
    return frame.rename(columns={'session_prediction': 'session6'})


def context(start, stop):
    origins = np.flatnonzero((TIMES >= start) & (TIMES < stop) & (TIMES.hour == 10) & (TIMES.minute == 30))
    expected_days = (stop - start).days
    if len(origins) != expected_days or origins[-1] + max(HORIZONS) >= len(TIMES):
        raise ValueError('Не хватает дней или целевых горизонтов')
    y = VALUES[TARGET]
    windows = np.arange(-47, 1)
    level24 = np.stack([y[pos + windows].mean(axis=0, dtype=np.float64) for pos in origins])
    seasonal_origins = origins[:, None] - 48 * np.arange(1, 29)
    if seasonal_origins.min() + windows.min() < 0:
        raise ValueError('Не хватает наблюдений для profile28')
    seasonal_level = np.stack([y[pos + windows].mean(axis=0, dtype=np.float64)
                               for pos in seasonal_origins.ravel()]).reshape(*seasonal_origins.shape, len(ROUTES))
    ratio = level24[:, None, :] / np.maximum(seasonal_level, 1)
    route_count = len(ROUTES)
    common = dict(cutoff=TIMES[origins].repeat(route_count), route_id=np.tile(ROUTES, len(origins)),
                  level24=level24.ravel(), current=y[origins].ravel(),
                  delta1h=(y[origins] - y[origins - 2]).ravel(),
                  delta2h=(y[origins] - y[origins - 4]).ravel(),
                  network_current=np.repeat(y[origins].sum(axis=1, dtype=np.float64), route_count))
    return pd.concat([pd.DataFrame(dict(common, horizon=h, y=y[origins + h].ravel(),
                                        profile28=(y[seasonal_origins + h] * ratio).mean(axis=1).ravel()))
                      for h in HORIZONS], ignore_index=True).sort_values(KEYS).reset_index(drop=True)


def join_base(frame, incoming, column):
    relevant = incoming[KEYS + ['y', column]]
    merged = frame.merge(relevant, on=KEYS, how='left', validate='one_to_one', suffixes=('', '_incoming'))
    if len(relevant) != len(frame) or not np.array_equal(merged.y, merged.y_incoming):
        raise ValueError(f'Не совпали ключи или факты для {column}')
    if not np.isfinite(merged[column]).all() or merged[column].lt(0).any():
        raise ValueError(f'Некорректные прогнозы {column}')
    return merged.drop(columns='y_incoming')


def assemble(start, stop, session, rf, cat, xgb, xgb_clean):
    frame = context(start, stop)
    for source, name in [(session, 'session6'), (rf, 'rf'), (cat, 'cat_rich'),
                         (xgb, 'xgb_rich'), (xgb_clean, 'xgb_clean')]:
        part = source.loc[source.cutoff.ge(start) & source.cutoff.lt(stop)]
        frame = join_base(frame, part, name)
    return frame


def archive_report():
    path = ARTIFACTS.with_suffix('.zip')
    temp = path.with_suffix('.partial')
    with zipfile.ZipFile(temp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=2) as archive:
        for file in sorted(ARTIFACTS.rglob('*')):
            if file.is_file() and not file.name.endswith('.partial'):
                archive.write(file, file.relative_to(ARTIFACTS.parent))
    temp.replace(path)
    print('Архив результатов', path)


In [ ]:
import joblib
import torch
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor


PAIRS = BASES[1:]
VOLUME_INPUTS = [name for name in PAIRS if name != 'xgb_clean']
SHAPES = ['session6', 'equal', 'joint', 'joint_without_xgb_clean', 'by_horizon',
          *[f'pair_{name}' for name in PAIRS],
          'ridge_100', 'ridge_1000', 'cat_d2', 'cat_d4', 'xgb_d2', 'xgb_d3', 'gating_8', 'gating_16']
VOLUMES = ['fixed', 'ratio', 'ewma7', 'ridge']
META_DEVICE = 'cuda'


def with_volume(frame, values, anchor=None):
    result = np.maximum(np.asarray(values, dtype=np.float64), 0)
    reference = frame.session6.to_numpy(dtype=np.float64) if anchor is None else np.asarray(anchor, dtype=np.float64)
    totals = frame[GROUP_KEYS].assign(pred=result, reference=reference).groupby(GROUP_KEYS, sort=False).transform('sum')
    reference_total = totals.reference.to_numpy()
    prediction_total = totals.pred.to_numpy()
    out = np.divide(result * reference_total, prediction_total, out=reference.copy(), where=prediction_total > 0)
    predicted = frame[GROUP_KEYS].assign(value=out).groupby(GROUP_KEYS, sort=False).value.sum()
    expected = frame[GROUP_KEYS].assign(value=reference).groupby(GROUP_KEYS, sort=False).value.sum()
    if not np.isfinite(out).all() or not np.allclose(predicted, expected, rtol=1e-10, atol=1e-5):
        raise ValueError('Не удалось сохранить заданный объём')
    return out


def shape_matrix(frame):
    return np.column_stack([with_volume(frame, frame[name]) for name in BASES])


def fit_weights(x, y):
    y = np.asarray(y, dtype=np.float64)
    denominator = y.sum()
    if denominator <= 0:
        raise ValueError('Нулевая сумма фактов для подбора весов')
    weights = np.zeros(x.shape[1])
    weights[0] = 1
    prediction = x[:, 0].copy()
    best = np.abs(prediction - y).sum() / denominator
    for step in (.2, .05):
        for _ in range(20):
            choice = None
            for source in np.flatnonzero(weights >= step - 1e-10):
                for destination in range(len(weights)):
                    if source == destination:
                        continue
                    amount = min(step, weights[source])
                    trial = prediction + amount * (x[:, destination] - x[:, source])
                    loss = np.abs(trial - y).sum() / denominator
                    if loss < best - 1e-9:
                        best = loss
                        choice = source, destination, amount, trial
            if choice is None:
                break
            source, destination, amount, prediction = choice
            weights[source] -= amount
            weights[destination] += amount
    return weights


def meta_features(frame, matrix, baseline):
    level = np.maximum(frame.level24.to_numpy(dtype=np.float64), 1)
    context = pd.DataFrame(dict(
        log_level=np.log1p(level), current=frame.current.to_numpy() / level,
        delta1h=frame.delta1h.to_numpy() / level, delta2h=frame.delta2h.to_numpy() / level,
        log_network=np.log1p(frame.network_current.to_numpy()),
        horizon=frame.horizon.to_numpy(), dayofweek=frame.cutoff.dt.day_of_week.to_numpy(),
        shape_disagreement=matrix.std(axis=1) / level, blend=baseline / level))
    for index, name in enumerate(BASES):
        context[f'{name}_at_anchor_volume'] = matrix[:, index] / level
        context[f'{name}_minus_session6'] = (matrix[:, index] - matrix[:, 0]) / level
    return context.clip(-50, 50).to_numpy(dtype=np.float32), level


class GatingNetwork(torch.nn.Module):
    def __init__(self, width, experts, hidden, prior):
        super().__init__()
        self.layers = torch.nn.Sequential(torch.nn.Linear(width, hidden), torch.nn.Tanh(),
                                          torch.nn.Linear(hidden, experts))
        self.register_buffer('prior', torch.as_tensor(prior, dtype=torch.float32))
        torch.nn.init.zeros_(self.layers[-1].weight)
        torch.nn.init.zeros_(self.layers[-1].bias)

    def forward(self, context):
        return torch.softmax(self.prior.log() + self.layers(context), dim=1)


def fit_gating(x, experts, y, level, prior, hidden):
    torch.manual_seed(SEED)
    scaler = StandardScaler().fit(x)
    prior = (prior + .02) / (prior.sum() + .02 * len(prior))
    model = GatingNetwork(x.shape[1], experts.shape[1], hidden, prior).to(META_DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=.003, weight_decay=.01)
    normalized = experts / level[:, None]
    target = np.asarray(y, dtype=np.float64) / level
    weights = level / level.mean()
    tensors = [torch.as_tensor(value, dtype=torch.float32, device=META_DEVICE)
               for value in (scaler.transform(x), normalized, target, weights)]
    tx, te, ty, tw = tensors
    for _ in range(15):
        permutation = torch.randperm(len(x), device=META_DEVICE)
        for rows in permutation.split(4096):
            gate = model(tx[rows])
            pred = (gate * te[rows]).sum(dim=1)
            loss = ((pred - ty[rows]).abs() * tw[rows]).mean() + .03 * (gate - model.prior).square().sum(dim=1).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return scaler, model.cpu()


def fit_shape(name, frame, matrix, joint):
    if name == 'session6':
        return None
    if name == 'equal':
        return np.full(len(BASES), 1 / len(BASES))
    if name == 'joint':
        return joint
    if name == 'joint_without_xgb_clean':
        mask = np.array([base != 'xgb_clean' for base in BASES])
        weights = np.zeros(len(BASES))
        weights[mask] = fit_weights(matrix[:, mask], frame.y)
        return weights
    if name.startswith('pair_'):
        index = BASES.index(name.removeprefix('pair_'))
        weights = np.zeros(len(BASES))
        weights[[0, index]] = fit_weights(matrix[:, [0, index]], frame.y)
        return weights
    if name == 'by_horizon':
        weights = []
        for h in HORIZONS:
            mask = frame.horizon.eq(h).to_numpy()
            weights.append(.5 * joint + .5 * fit_weights(matrix[mask], frame.y.to_numpy()[mask]))
        return np.stack(weights)
    baseline = matrix @ joint
    features, level = meta_features(frame, matrix, baseline)
    residual = (frame.y.to_numpy(dtype=np.float64) - baseline) / level
    sample_weight = level / level.mean()
    if name.startswith('gating_'):
        return fit_gating(features, matrix, frame.y, level, joint, int(name.split('_')[-1]))
    if name.startswith('ridge_'):
        prep = StandardScaler().fit(features)
        model = Ridge(alpha=float(name.split('_')[-1]))
        model.fit(prep.transform(features), residual, sample_weight=sample_weight)
        return prep, model
    depth = int(name[-1])
    if name.startswith('cat_'):
        model = CatBoostRegressor(depth=depth, iterations=220, loss_function='MAE', learning_rate=.035,
                                  l2_leaf_reg=20, random_seed=SEED, verbose=False, allow_writing_files=False,
                                  thread_count=THREADS)
    else:
        model = XGBRegressor(max_depth=depth, n_estimators=220, objective='reg:absoluteerror',
                             tree_method='hist', learning_rate=.035, reg_lambda=20,
                             min_child_weight=30, random_state=SEED, n_jobs=THREADS)
    model.fit(features, residual, sample_weight=sample_weight)
    return None, model


def predict_shape(name, fitted, frame, matrix, joint):
    if name == 'session6':
        return matrix[:, 0]
    if isinstance(fitted, np.ndarray):
        if fitted.ndim == 1:
            return matrix @ fitted
        return np.sum(matrix * fitted[frame.horizon.to_numpy() - 1], axis=1)
    baseline = matrix @ joint
    prep, model = fitted
    features, level = meta_features(frame, matrix, baseline)
    if name.startswith('gating_'):
        weights = gate_weights(prep, model, features)
        return with_volume(frame, np.sum(matrix * weights, axis=1))
    encoded = features if prep is None else prep.transform(features)
    correction = np.clip(model.predict(encoded), -.5, .5) * level
    return with_volume(frame, baseline + correction)


def gate_weights(prep, model, features):
    model.eval()
    with torch.no_grad():
        return model(torch.as_tensor(prep.transform(features), dtype=torch.float32)).numpy()


def volume_table(frame, with_y=False):
    aggregations = {name: (name, 'sum') for name in BASES}
    aggregations['network_current'] = ('network_current', 'first')
    if with_y:
        aggregations['y'] = ('y', 'sum')
    return frame.groupby(GROUP_KEYS, as_index=False, sort=True).agg(**aggregations)


def volume_features(table):
    anchor = np.maximum(table.session6.to_numpy(), 1)
    features = np.column_stack([np.log1p(anchor), np.log1p(table.network_current),
                                table.horizon, table.cutoff.dt.day_of_week,
                                *[np.log((table[name].to_numpy() + 1) / (anchor + 1)) for name in VOLUME_INPUTS]])
    return features


def fit_volumes(frame):
    table = volume_table(frame, with_y=True)
    if table.session6.sum() <= 0:
        raise ValueError('Нулевой объём Session6')
    ages = (table.cutoff.max() - table.cutoff).dt.total_seconds().to_numpy() / 86400
    w = np.exp2(-ages / 7)
    ratio = table.y.sum() / table.session6.sum()
    recent_ratio = np.dot(table.y, w) / np.dot(table.session6, w)
    x = volume_features(table)
    prep = StandardScaler().fit(x)
    model = Ridge(alpha=100).fit(prep.transform(x), np.log((table.y + 1) / (table.session6 + 1)))
    return dict(fixed=1., ratio=float(1 + .5 * (ratio - 1)),
                ewma7=float(1 + .5 * (recent_ratio - 1)), ridge=(prep, model))


def volume_multipliers(frame, fitted):
    table = volume_table(frame)
    prep, model = fitted['ridge']
    fitted = dict(fitted, ridge=1 + .5 * np.expm1(model.predict(prep.transform(volume_features(table)))))
    return {name: frame[GROUP_KEYS].merge(table[GROUP_KEYS].assign(factor=np.clip(factor, .95, 1.05)),
                                           on=GROUP_KEYS, how='left', validate='many_to_one').factor.to_numpy()
            for name, factor in fitted.items()}


def fit_meta(train, query, phase, split):
    train_matrix, query_matrix = shape_matrix(train), shape_matrix(query)
    joint = fit_weights(train_matrix, train.y)
    volume_models = fit_volumes(train)
    train_factors = volume_multipliers(train, volume_models)
    query_factors = volume_multipliers(query, volume_models)
    rows, fitted, predictions = [], {}, {}
    for name in SHAPES:
        model = fit_shape(name, train, train_matrix, joint)
        fitted[name] = model
        train_prediction = predict_shape(name, model, train, train_matrix, joint)
        query_prediction = predict_shape(name, model, query, query_matrix, joint)
        local = []
        for volume in VOLUMES:
            label = f'{name}__{volume}'
            train_values = train_prediction * train_factors[volume]
            query_values = query_prediction * query_factors[volume]
            predictions[label] = query_values
            local.extend([dict(phase=phase, variant=label, shape=name, volume=volume, period=period, **metric(y, p))
                          for period, y, p in [('train_oof', train.y, train_values), (split, query.y, query_values)]])
        display(pd.DataFrame(local))
        rows.extend(local)
    return pd.DataFrame(rows), dict(joint=joint, shapes=fitted, volumes=volume_models), predictions


def fit_selection(oof):
    blocks = []
    for fold, (start, stop) in enumerate(OOF_WINDOWS[2:], 3):
        train = oof.loc[oof.cutoff.lt(start)]
        validation = oof.loc[oof.cutoff.ge(start) & oof.cutoff.lt(stop)]
        rows, _, _ = fit_meta(train, validation, f'val_week_{fold}', 'val')
        blocks.append(rows)
    metrics = pd.concat(blocks, ignore_index=True)
    metrics.to_csv(ARTIFACTS / 'selection_metrics.csv', index=False)
    ranking = metrics.loc[metrics.period.eq('val')].groupby(['variant', 'shape', 'volume'], as_index=False).agg(
        mean_val_score=('score', 'mean'), worst_val_score=('score', 'max'), weeks=('score', 'count'))
    ranking['order'] = ranking['shape'].map({name: i for i, name in enumerate(SHAPES)})
    ranking = ranking.sort_values(['mean_val_score', 'worst_val_score', 'order', 'variant'])
    if not ranking.weeks.eq(2).all():
        raise ValueError('В подборе должны присутствовать обе недели val')
    ranking.to_csv(ARTIFACTS / 'selection_ranking.csv', index=False)
    display(ranking.drop(columns='order').head(20))
    selected = ranking.iloc[0]
    frozen = dict(variant=selected.variant, shape=selected['shape'], volume=selected.volume,
                  mean_val_score=float(selected.mean_val_score), test_used_for_selection=False,
                  meta_fit_weeks=4, local_test_before_full_refit=True)
    write_json(ARTIFACTS / 'frozen_selection.json', frozen)
    return frozen


def evaluate(oof, test, frozen):
    rows, models, predictions = fit_meta(oof, test, 'frozen_local_test', 'test')
    base_rows = [dict(phase='base', variant=column, shape=column, volume='raw', period=period,
                      **metric(frame.y, frame[column]))
                 for period, frame in [('train_oof', oof), ('test', test)] for column in BASES]
    table = pd.concat([pd.DataFrame(base_rows), rows], ignore_index=True)
    table['selected_on_val'] = table.variant.eq(frozen['variant'])
    table.to_csv(ARTIFACTS / 'train_test_metrics.csv', index=False)
    predictions = pd.concat([test.reset_index(drop=True), pd.DataFrame(predictions)], axis=1)
    save_parquet(ARTIFACTS / 'test_predictions.parquet', predictions)
    columns = [*BASES, *[f'{name}__fixed' for name in SHAPES]]
    if frozen['variant'] not in columns:
        columns.append(frozen['variant'])
    slices = []
    for kind, key in [('day', predictions.cutoff), ('horizon', predictions.horizon),
                      ('week', ((predictions.cutoff.dt.normalize() - LOCAL_TEST_START).dt.days // 7) + 1)]:
        for value, group in predictions.groupby(key):
            slices.extend(dict(slice=kind, value=str(value), variant=name, **metric(group.y, group[name])) for name in columns)
    pd.DataFrame(slices).to_csv(ARTIFACTS / 'test_slices.csv', index=False)
    weights = []
    matrix = shape_matrix(test)
    features, _ = meta_features(test, matrix, matrix @ models['joint'])
    for name, model in models['shapes'].items():
        if isinstance(model, np.ndarray):
            if model.ndim == 1:
                weights.append(dict(variant=name, horizon='all', **dict(zip(BASES, model))))
            else:
                weights.extend(dict(variant=name, horizon=h, **dict(zip(BASES, row)))
                               for h, row in zip(HORIZONS, model))
        elif name.startswith('gating_'):
            prep, network = model
            frame = pd.DataFrame(gate_weights(prep, network, features), columns=BASES)
            means = frame.groupby(test.horizon.to_numpy()).mean()
            weights.extend(dict(variant=name, horizon=h, **row.to_dict()) for h, row in means.iterrows())
            models['shapes'][name] = dict(prep=prep, state=network.state_dict(),
                                         width=features.shape[1], hidden=int(name.split('_')[-1]))
    pd.DataFrame(weights).to_csv(ARTIFACTS / 'blend_weights.csv', index=False)
    artifact = ARTIFACTS / 'evaluation_meta.joblib'
    temporary = artifact.with_suffix('.partial')
    joblib.dump(dict(frozen=frozen, bases=BASES, models=models), temporary)
    temporary.replace(artifact)
    comparison = ['joint__fixed', 'joint_without_xgb_clean__fixed', 'pair_xgb_clean__fixed']
    display(table.loc[table.selected_on_val | table.variant.isin([*BASES, *comparison])])
    note = ('Независимая от полного переобучения оценка test получена до переобучения. '
            'Локальный период 18–31 октября уже изучался. Выбор сделан на двух более ранних неделях. '
            'Новые метамодели обучены только на OOF до test. '
            'Session6 на OOF восстановлен по рецепту, а на test взят из архива решения 0.326. '
            'Полное переобучение и конкурсный сабмит не выполняются.')
    (ARTIFACTS / 'evaluation_note.txt').write_text(note)
    print(note)


In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '4'
import torch
import joblib
from sklearn.preprocessing import StandardScaler


def find_previous_run(explicit=None):
    required = ['base_oof.parquet', 'test_predictions.parquet', 'frozen_selection.json']
    candidates = [Path(explicit)] if explicit else sorted({p.parent for p in Path('/kaggle/input').rglob('base_oof.parquet')
        if all((p.parent / f).is_file() for f in required)})
    if len(candidates) != 1:
        raise ValueError(f'Укажите EVALUATION_DIR: {candidates}')
    return candidates[0]


def fit_candidate(name, frame):
    matrix = shape_matrix(frame)
    joint = fit_weights(matrix, frame.y)
    if name == 'cat_d4__ridge':
        return dict(name=name, joint=joint, shape=fit_shape('cat_d4', frame, matrix, joint), volumes=fit_volumes(frame))
    features, level = meta_features(frame, matrix, matrix @ joint)
    prep = StandardScaler().fit(features)
    torch.manual_seed(SEED)
    width = features.shape[1]
    model = torch.nn.Linear(width, 1) if name.startswith('linear') else torch.nn.Sequential(
        torch.nn.Linear(width, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))
    last = model if isinstance(model, torch.nn.Linear) else model[-1]
    torch.nn.init.zeros_(last.weight)
    torch.nn.init.zeros_(last.bias)
    x = torch.as_tensor(prep.transform(features), dtype=torch.float32)
    scale = max(float(np.mean(frame.y)), 1.)
    y = torch.as_tensor(frame.y.to_numpy() / scale, dtype=torch.float32)
    anchor = torch.as_tensor((matrix @ joint) / scale, dtype=torch.float32)
    optimizer = torch.optim.AdamW(model.parameters(), lr=.01, weight_decay=.01)
    losses = []
    for epoch in range(160):
        correction = .05 * torch.tanh(model(x).ravel())
        prediction = anchor * (1 + correction)
        error = prediction - y
        loss = error.abs().sum() / y.sum()
        if name.endswith('metric'):
            loss = loss + error.sum().abs() / y.sum()
        loss = loss + .1 * correction.square().mean()
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
        optimizer.step()
        losses.append(float(loss.detach()))
    return dict(name=name, joint=joint, prep=prep, model=model.eval(), loss=losses)


def predict_candidate(fitted, frame):
    matrix = shape_matrix(frame)
    if fitted['name'] == 'cat_d4__ridge':
        return predict_shape('cat_d4', fitted['shape'], frame, matrix, fitted['joint']) * volume_multipliers(frame, fitted['volumes'])['ridge']
    features, level = meta_features(frame, matrix, matrix @ fitted['joint'])
    x = torch.as_tensor(fitted['prep'].transform(features), dtype=torch.float32)
    with torch.no_grad():
        correction = .05 * torch.tanh(fitted['model'](x).ravel()).numpy()
    return matrix @ fitted['joint'] * (1 + correction)


def verify_frame(frame):
    needed = [*BASES, 'y', 'level24', 'current', 'delta1h', 'delta2h', 'network_current']
    if frame.duplicated(KEYS).any() or not np.isfinite(frame[needed]).all().all() or frame.y.lt(0).any():
        raise ValueError('Некорректные исторические прогнозы')
    groups = frame.groupby(GROUP_KEYS).route_id.nunique()
    if not groups.eq(frame.route_id.nunique()).all():
        raise ValueError('Неполные группы маршрутов')


def experiment_select(oof):
    rows = []
    for fold, (start, stop) in enumerate(OOF_WINDOWS[2:], 3):
        train = oof.loc[(oof.cutoff + oof.horizon * STEP).lt(start)].reset_index(drop=True)
        val = oof.loc[oof.cutoff.ge(start) & oof.cutoff.lt(stop)].reset_index(drop=True)
        for name in CANDIDATES:
            model = fit_candidate(name, train)
            local = [dict(fold=fold, candidate=name, period=period, **metric(frame.y, predict_candidate(model, frame)))
                     for period, frame in [('train_oof', train), ('val', val)]]
            display(pd.DataFrame(local))
            rows.extend(local)
    table = pd.DataFrame(rows)
    table.to_csv(ARTIFACTS / 'validation_metrics.csv', index=False)
    ranking = table.loc[table.period.eq('val')].groupby('candidate', as_index=False).agg(
        mean_score=('score', 'mean'), worst_score=('score', 'max'))
    ranking['order'] = ranking.candidate.map({n: i for i, n in enumerate(CANDIDATES)})
    ranking = ranking.sort_values(['mean_score', 'worst_score', 'order'])
    ranking.to_csv(ARTIFACTS / 'validation_ranking.csv', index=False)
    frozen = dict(candidate=ranking.iloc[0].candidate, test_used_for_selection=False,
                  epochs=160, correction_bound=.05, candidates=CANDIDATES)
    write_json(ARTIFACTS / 'frozen_selection.json', frozen)
    display(ranking)
    return frozen


def experiment_test(oof, test, frozen):
    rows, predictions = [], test[KEYS + ['y']].copy()
    models = {}
    for name in dict.fromkeys(['cat_d4__ridge', frozen['candidate']]):
        model = fit_candidate(name, oof)
        models[name] = model
        local = []
        for period, frame in [('train_oof', oof), ('test', test)]:
            prediction = predict_candidate(model, frame)
            local.append(dict(candidate=name, selected=name == frozen['candidate'], period=period, **metric(frame.y, prediction)))
            if period == 'test':
                predictions[name] = prediction
        rows.extend(local)
        display(pd.DataFrame(local))
    pd.DataFrame(rows).to_csv(ARTIFACTS / 'train_test_metrics.csv', index=False)
    save_parquet(ARTIFACTS / 'test_predictions.parquet', predictions)
    daily = []
    for day, frame in predictions.groupby(predictions.cutoff.dt.normalize()):
        for name in models:
            daily.append(dict(day=str(day.date()), candidate=name, **metric(frame.y, frame[name])))
    pd.DataFrame(daily).to_csv(ARTIFACTS / 'daily_metrics.csv', index=False)
    joblib.dump(models, ARTIFACTS / 'evaluation_models.joblib')
    note = ('Оценка test получена до полного переобучения. Test 18–31 октября уже изучался ранее. '
            'Кандидат выбран на двух предтестовых неделях. Train OOF — собственное обучение метамодели. '
            'Новые базы не обучались, конкурсный сабмит не создавался. Для сабмита нужен полный refit всех компонентов.')
    (ARTIFACTS / 'evaluation_note.txt').write_text(note)
    print(note)
    archive_report()


In [ ]:
CODE_HASH = 'e4e9f44a05505a567be27d21fe6a2be03e55891ba43e81ba0185ca161b5d219e'
EVALUATION_DIR = None
torch.set_num_threads(4)
EVALUATION = find_previous_run(EVALUATION_DIR)
CANDIDATES = ['cat_d4__ridge', 'linear_mae', 'linear_metric', 'mlp_mae', 'mlp_metric']
contract = dict(code=CODE_HASH, oof=digest_file(EVALUATION / 'base_oof.parquet'))
run_id = hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()[:12]
ARTIFACTS = Path('/kaggle/working') / f'blend_metric_{run_id}'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
write_json(ARTIFACTS / 'inputs.json', dict(contract, source=str(EVALUATION)))
oof = pd.read_parquet(EVALUATION / 'base_oof.parquet')
verify_frame(oof)
if oof.cutoff.min() != OOF_START + pd.Timedelta(hours=10, minutes=30) or not (oof.cutoff + oof.horizon * STEP).lt(LOCAL_TEST_START).all():
    raise ValueError('Неверный период OOF')
frozen = experiment_select(oof)


In [ ]:
if json.loads((ARTIFACTS / 'frozen_selection.json').read_text()) != frozen:
    raise ValueError('Выбор должен быть зафиксирован до test')
test = pd.read_parquet(EVALUATION / 'test_predictions.parquet')
verify_frame(test)
if not test.cutoff.ge(LOCAL_TEST_START).all() or not (test.cutoff + test.horizon * STEP).lt(LOCAL_TEST_END).all():
    raise ValueError('Неверный период test')
experiment_test(oof, test, frozen)
